In [0]:
from pyspark.sql import functions as F

# =========================================================
# CONFIGURATION
# =========================================================

SEED = 42
BASE_PATH = "/Volumes/workspace/finance_analytics/finance_raw"

# =========================================================
# LOAD SOURCE DATA
# =========================================================

customers_df = spark.read.parquet(f"{BASE_PATH}/customers")
products_df = spark.read.parquet(f"{BASE_PATH}/products")
branches_df = spark.read.parquet(f"{BASE_PATH}/branches")

print(f"Customers loaded: {customers_df.count():,}")
print(f"Products loaded: {products_df.count():,}")
print(f"Branches loaded: {branches_df.count():,}")

# =========================================================
# GENERATE LOANS
# =========================================================

loans_df = (
    spark.range(1, 8001)
    .withColumnRenamed("id", "loan_id")
    
    # Link loan to an existing customer
    .withColumn(
        "customer_id",
        (F.rand(SEED) * 10000).cast("int") + 1
    )
    
    # Loan products: IDs 4-8 and 10
    .withColumn(
        "product_id",
        F.when(F.rand(SEED + 1) < 0.20, 4)
         .when(F.rand(SEED + 2) < 0.40, 5)
         .when(F.rand(SEED + 3) < 0.60, 6)
         .when(F.rand(SEED + 4) < 0.75, 7)
         .when(F.rand(SEED + 5) < 0.90, 8)
         .otherwise(10)
    )
    
    # Link loan to an existing branch
    .withColumn(
        "branch_id",
        (F.rand(SEED + 6) * 30).cast("int") + 1
    )
    
    # Loan amount between €5,000 and €250,000
    .withColumn(
        "loan_amount",
        F.round(
            F.rand(SEED + 7) * 245000 + 5000,
            2
        )
    )
    
    # Interest rate between 4% and 10%
    .withColumn(
        "interest_rate",
        F.round(
            F.rand(SEED + 8) * 6 + 4,
            2
        )
    )
    
    # Loan start date between 2024-01-01 and 2025-12-31
    .withColumn(
        "start_date",
        F.date_add(
            F.to_date(F.lit("2024-01-01")),
            (F.rand(SEED + 9) * 731).cast("int")
        )
    )
    
    # Maturity date 1-10 years after start
    .withColumn(
        "maturity_date",
        F.add_months(
            F.col("start_date"),
            ((F.rand(SEED + 10) * 108) + 12).cast("int")
        )
    )
    
    # Loan status
    .withColumn(
        "loan_status",
        F.when(F.rand(SEED + 11) < 0.72, "Active")
         .when(F.rand(SEED + 11) < 0.88, "Closed")
         .when(F.rand(SEED + 11) < 0.96, "Delinquent")
         .otherwise("Default")
    )
    
    # Outstanding balance based on loan status
    .withColumn(
        "outstanding_balance",
        F.when(
            F.col("loan_status") == "Closed",
            F.lit(0.0)
        )
        .otherwise(
            F.round(
                F.col("loan_amount") *
                (F.rand(SEED + 12) * 0.90 + 0.05),
                2
            )
        )
    )
    
    # Credit score at origination
    .withColumn(
        "credit_score_at_origination",
        (F.rand(SEED + 13) * 301 + 500).cast("int")
    )
)

# Select final columns in business-friendly order
loans_df = loans_df.select(
    "loan_id",
    "customer_id",
    "product_id",
    "branch_id",
    "loan_amount",
    "outstanding_balance",
    "interest_rate",
    "start_date",
    "maturity_date",
    "loan_status",
    "credit_score_at_origination"
)

display(loans_df.limit(20))

In [0]:
# =========================================================
# LOAN DATA VALIDATION
# =========================================================

print(f"Loan count: {loans_df.count():,}")

print("Loan status distribution:")
display(
    loans_df.groupBy("loan_status")
    .count()
    .orderBy("loan_status")
)

print("Checking invalid customer references...")

invalid_customers = (
    loans_df
    .join(
        customers_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print(f"Invalid customer references: {invalid_customers.count()}")

print("Checking invalid branch references...")

invalid_branches = (
    loans_df
    .join(
        branches_df.select("branch_id"),
        on="branch_id",
        how="left_anti"
    )
)

print(f"Invalid branch references: {invalid_branches.count()}")

print("Checking invalid product references...")

invalid_products = (
    loans_df
    .join(
        products_df.select("product_id"),
        on="product_id",
        how="left_anti"
    ))

print(f"Invalid product references: {invalid_products.count()}")

In [0]:
# =========================================================
# SAVE LOAN DATA TO FINANCE RAW VOLUME
# =========================================================

loans_df.write.mode("overwrite").parquet(
    f"{BASE_PATH}/loans"
)

print("Loan data successfully written to finance_raw Volume.")

In [0]:
# =========================================================
# VERIFY SAVED LOAN DATA
# =========================================================

saved_loans_df = spark.read.parquet(
    f"{BASE_PATH}/loans"
)

print(f"Saved loans: {saved_loans_df.count():,}")

display(saved_loans_df.limit(10))